# Hopfield

### Test

In [1]:
import sys
import os

PROJECT_ROOT = r"C:\Users\tom39\Desktop\MHNfs"

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("Path added.")

Path added.


In [2]:
from hopfield import Hopfield

c:\Users\tom39\Documents\.venv\.venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [9]:
import torch

hopfield = Hopfield(
    input_size=1024,
    hidden_size=512,
    stored_pattern_size=1024,
    pattern_projection_size=1024,
    output_size=1024,
    num_heads=4
)

context = torch.randn(8, 64, 1024)
state = torch.randn(8, 10, 1024)

out = hopfield((context, state, context))

print(out.shape)

torch.Size([8, 10, 1024])


In [10]:
import torch.nn.functional as F

def mean_similarity(x, memory):
    x_mean = x.mean(dim=1)
    mem_mean = memory.mean(dim=1)
    return F.cosine_similarity(x_mean, mem_mean).mean()

before = mean_similarity(state.detach(), context.detach())
after  = mean_similarity(out.detach(), context.detach())

print("Similarity BEFORE:", before.item())
print("Similarity AFTER :", after.item())

Similarity BEFORE: -0.0003987094387412071
Similarity AFTER : 0.01759161427617073


In [11]:
for beta in [0.1, 0.5, 1.0, 2.0]:
    hopfield = Hopfield(
        input_size=1024,
        hidden_size=512,
        stored_pattern_size=1024,
        pattern_projection_size=1024,
        output_size=1024,
        num_heads=4,
        scaling=beta
    )

    out = hopfield((context.detach(), state.detach(), context.detach()))

    movement = (out - state.detach()).norm(dim=-1).mean()
    print(f"beta={beta:.2f} | movement={movement.item():.4f}")

beta=0.10 | movement=32.1877
beta=0.50 | movement=34.5626
beta=1.00 | movement=35.7653
beta=2.00 | movement=36.3102


In [12]:
print("NaNs:", torch.isnan(out).any().item())
print("Max value:", out.abs().max().item())

NaNs: False
Max value: 2.591696262359619


### New Test code

In [259]:
import sys
sys.modules.pop("hopfield.my_hopfield", None)

<module 'hopfield.my_hopfield' from 'c:\\Users\\tom39\\Desktop\\MHNfs\\hopfield\\my_hopfield.py'>

In [260]:
import torch
import torch.nn.functional as F
from hopfield.my_hopfield import MyHopfield

torch.manual_seed(0)


In [261]:
# -----------------------------
# TEST SCRIPT
# -----------------------------

B = 4
Ns = 1
Nc = 10
D = 128

# Create Hopfield model
model = MyHopfield(input_size=D, num_heads=4, init_beta=5.0)

# Synthetic data
query = torch.randn(B, Ns, D, requires_grad=True)
context = torch.randn(B, Nc, D, requires_grad=True)

# -----------------------------
# TEST 1 — CROSS ATTENTION
# -----------------------------
out = model(query, context)
print("=== SHAPES ===")
print("query:", query.shape)
print("context:", context.shape)
print("output:", out.shape)

# -----------------------------
# TEST 2 — RETRIEVAL EFFECT
# -----------------------------
def cosine(q, c):
    q = q.squeeze(1)
    sims = F.cosine_similarity(q.unsqueeze(1), c, dim=-1)
    return sims.mean()

before = cosine(query.detach(), context.detach())
after = cosine(out.detach(), context.detach())
print("=== CONTEXT SIMILARITY ===")
print("before:", before.item())
print("after:", after.item())

# -----------------------------
# TEST 3 — BETA SENSITIVITY
# -----------------------------
print("=== BETA SWEEP ===")
for beta in [0.1, 0.5, 1.0, 2.0, 5.0]:
    model.beta_param.data.fill_(beta)  # <<< FIXED!
    out = model(query, context)
    movement = (out - query).norm(dim=-1).mean()
    print(f"beta={beta:.2f}  movement={movement.item():.4f}")

# -----------------------------
# TEST 4 — ASSOCIATION MATRIX
# -----------------------------
print("=== ASSOCIATION MATRIX ===")
A = model.get_association_matrix(query, context)
print("shape:", A.shape)
print("row sums (should be 1):")
print(A.sum(-1))

# -----------------------------
# TEST 5 — GRADIENT FLOW
# -----------------------------
loss = out.mean()
loss.backward()
print("\n=== GRADIENT NORMS ===")
print("query grad:", query.grad.norm().item())
print("context grad:", context.grad.norm().item())

model.eval()

# -----------------------------
# MEMORY RETRIEVAL
# -----------------------------
print("===========================================")
print("Retrieved Memory Test")

B, M, D = 1, 10, 128
memory = torch.randn(B, M, D)
query = memory[:, 3:4, :] + 0.1*torch.randn(B, 1, D)

model = MyHopfield(input_size=D, num_heads=4, init_beta=5.0)
out = model(query, memory)

sim_before = F.cosine_similarity(query, memory[:, 3:4, :], dim=-1).item()
sim_after  = F.cosine_similarity(out, memory[:, 3:4, :], dim=-1).item()
print("before:", sim_before)
print("after: ", sim_after)

sim = F.cosine_similarity(out, memory, dim=-1)
print("Memory Index:", sim.argmax())

=== SHAPES ===
query: torch.Size([4, 1, 128])
context: torch.Size([4, 10, 128])
output: torch.Size([4, 1, 128])
=== CONTEXT SIMILARITY ===
before: -0.008532954379916191
after: -0.0017771061975508928
=== BETA SWEEP ===
beta=0.10  movement=4.0784
beta=0.50  movement=3.9336
beta=1.00  movement=3.9196
beta=2.00  movement=3.7905
beta=5.00  movement=4.0541
=== ASSOCIATION MATRIX ===
shape: torch.Size([4, 4, 1, 10])
row sums (should be 1):
tensor([[[1.0000],
         [1.0000],
         [1.0000],
         [1.0000]],

        [[1.0000],
         [1.0000],
         [1.0000],
         [1.0000]],

        [[1.0000],
         [1.0000],
         [1.0000],
         [1.0000]],

        [[1.0000],
         [1.0000],
         [1.0000],
         [1.0000]]], grad_fn=<SumBackward1>)

=== GRADIENT NORMS ===
query grad: 0.04419996216893196
context grad: 0.01982998289167881
Retrieved Memory Test
before: 0.9934307932853699
after:  0.9179939031600952
Memory Index: tensor(3)


In [262]:
model.eval()

energy_before = model.compute_energy(query, context)

out = model(query, context)

energy_after = model.compute_energy(out, context)

print("energy before:", energy_before.mean().item())
print("energy after :", energy_after.mean().item())

x = query

for i in range(5):
    energy = model.compute_energy(x, context).mean().item()
    print(f"step {i}: energy={energy:.4f}")
    x = model(x, context)



energy before: -2.143407106399536
energy after : -2.1432344913482666
step 0: energy=-2.1434
step 1: energy=-2.1432
step 2: energy=-2.1431
step 3: energy=-2.1430
step 4: energy=-2.1429


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class MyHopfield(nn.Module):
    """
    Transformer-based associative memory layer (Modern Hopfield).

    Retrieval rule:
        softmax(beta * QK^T / sqrt(d)) V

    Query  = sample embedding
    Key    = stored memory patterns
    Value  = stored memory patterns
    """

    def __init__(
        self,
        input_size: int,
        num_heads: int = 8,
        dropout: float = 0.1,
        batch_first: bool = True,
        use_layer_norm: bool = True
    ):
        super().__init__()

        self.input_size = input_size
        self.num_heads = num_heads
        self.batch_first = batch_first
        self.use_layer_norm = use_layer_norm

        # Multi-head associative retrieval
        self.attention = nn.MultiheadAttention(
            embed_dim=input_size,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=batch_first
        )

        # Hopfield inverse temperature (learnable)
        self.beta_param = nn.Parameter(torch.tensor(0.0))

        # Optional normalization
        if self.use_layer_norm:
            self.norm_query = nn.LayerNorm(input_size)
            self.norm_key = nn.LayerNorm(input_size)
            self.norm_value = nn.LayerNorm(input_size)

        self.reset_parameters()

    # --------------------------------------------------
    # Positive inverse temperature
    # --------------------------------------------------

    @property
    def beta(self):
        return F.softplus(self.beta_param)

    # --------------------------------------------------
    # Forward retrieval
    # --------------------------------------------------

    def forward(
        self,
        query: torch.Tensor,
        key: torch.Tensor = None,
        value: torch.Tensor = None,
        stored_pattern_padding_mask: torch.Tensor = None,
        association_mask: torch.Tensor = None
    ):

        if key is None:
            key = query

        if value is None:
            value = key

        # Layer normalization
        if self.use_layer_norm:
            query = self.norm_query(query)
            key = self.norm_key(key)
            value = self.norm_value(value)

        # Hopfield temperature scaling
        query = self.beta * query

        output, _ = self.attention(
            query=query,
            key=key,
            value=value,
            attn_mask=association_mask,
            key_padding_mask=stored_pattern_padding_mask,
            need_weights=False
        )

        return output

    # --------------------------------------------------
    # Association matrix
    # --------------------------------------------------

    def get_association_matrix(
        self,
        query,
        key=None,
        value=None
    ):

        if key is None:
            key = query

        if value is None:
            value = key

        if self.use_layer_norm:
            query = self.norm_query(query)
            key = self.norm_key(key)
            value = self.norm_value(value)

        query = self.beta * query

        # IMPORTANT: disable averaging over heads
        _, attn_weights = self.attention(
            query=query,
            key=key,
            value=value,
            need_weights=True,
            average_attn_weights=False
        )

        # shape:
        # (B, heads, Q, M)

        # average heads manually
        attn_weights = attn_weights.mean(dim=1)

        # ensure rows sum to 1 (numerical stability)
        attn_weights = attn_weights / attn_weights.sum(dim=-1, keepdim=True)

        return attn_weights

    # --------------------------------------------------
    # Reset parameters
    # --------------------------------------------------

    def reset_parameters(self):

        self.attention._reset_parameters()

        if self.use_layer_norm:
            self.norm_query.reset_parameters()
            self.norm_key.reset_parameters()
            self.norm_value.reset_parameters()

        nn.init.constant_(self.beta_param, 0.0)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


class MyHopfield(nn.Module):

    def __init__(
        self,
        input_size,
        num_heads=4,
        use_layer_norm=True,
        use_projection=False,
        init_beta=1.0,
        attn_dropout=0.1,
        residual_dropout=0.1,
        beta_min=0.1,
        beta_max=10.0
    ):
        super().__init__()

        assert input_size % num_heads == 0

        self.input_size = input_size
        self.num_heads = num_heads
        self.head_dim = input_size // num_heads
        self.use_layer_norm = use_layer_norm
        self.use_projection = use_projection

        self.attn_dropout = nn.Dropout(attn_dropout)
        self.residual_dropout = nn.Dropout(residual_dropout)

        self.beta_min = beta_min
        self.beta_max = beta_max

        if use_projection:
            self.q_proj = nn.Linear(input_size, input_size)
            self.k_proj = nn.Linear(input_size, input_size)
            self.v_proj = nn.Linear(input_size, input_size)
            self.out_proj = nn.Linear(input_size, input_size)

        self.beta_param = nn.Parameter(torch.ones(num_heads) * init_beta)

        if use_layer_norm:
            self.norm_query = nn.LayerNorm(input_size)
            self.norm_key = nn.LayerNorm(input_size)
            self.norm_value = nn.LayerNorm(input_size)

        self.reset_parameters()

    # -------------------------------------------------
    # positive beta
    # -------------------------------------------------
    @property
    def beta(self):
        return F.softplus(self.beta_param)

    # -------------------------------------------------
    def split_heads(self, x):
        B, T, D = x.shape
        x = x.view(B, T, self.num_heads, self.head_dim)
        return x.transpose(1, 2)

    # -------------------------------------------------
    def merge_heads(self, x):
        B, H, T, Dh = x.shape
        x = x.transpose(1, 2).contiguous()
        return x.view(B, T, H * Dh)

    # -------------------------------------------------
    def _get_beta(self):

        beta = self.beta
        beta = torch.clamp(beta, self.beta_min, self.beta_max)

        if beta.numel() == 1:
            beta = beta.expand(self.num_heads)

        return beta.view(1, self.num_heads, 1, 1)

    # -------------------------------------------------
    def _apply_mask(self, scores, mask):

        if mask is None:
            return scores

        if mask.dim() == 2:
            mask = mask.unsqueeze(1).unsqueeze(1)

        elif mask.dim() == 3:
            mask = mask.unsqueeze(1)

        mask = mask.to(dtype=torch.bool)

        scores = scores.masked_fill(~mask, float("-inf"))

        return scores

    # -------------------------------------------------
    # single Hopfield update
    # -------------------------------------------------
    def forward(self, query, key=None, value=None, mask=None):

        residual = query

        if key is None:
            key = query
        if value is None:
            value = key

        if self.use_layer_norm:
            query = self.norm_query(query)
            key = self.norm_key(key)
            value = self.norm_value(value)

        if self.use_projection:
            q = self.q_proj(query)
            k = self.k_proj(key)
            v = self.v_proj(value)
        else:
            q = query
            k = key
            v = value

        q = self.split_heads(q)
        k = self.split_heads(k)
        v = self.split_heads(v)

        scores = torch.matmul(q, k.transpose(-2, -1))

        beta = self._get_beta()

        scores = scores * beta / math.sqrt(self.head_dim)

        scores = self._apply_mask(scores, mask)

        attn = F.softmax(scores, dim=-1)

        attn = self.attn_dropout(attn)

        out = torch.matmul(attn, v)

        out = self.merge_heads(out)

        if self.use_projection:
            out = self.out_proj(out)

        out = self.residual_dropout(out)

        out = residual + out

        return out

    # -------------------------------------------------
    # Iterative Retrieval
    # -------------------------------------------------
    def forward_iterative(
        self,
        query,
        key=None,
        value=None,
        mask=None,
        steps=3,
        tol=1e-4,
        return_energy=False
    ):

        state = query
        energies = []

        for _ in range(steps):

            prev = state

            state = self.forward(state, key, value, mask)

            if return_energy:
                energy = self.compute_energy(state, key)
                energies.append(energy.mean().item())

            # convergence check
            diff = torch.norm(state - prev) / torch.norm(prev)

            if diff < tol:
                break

        if return_energy:
            return state, energies

        return state

    # -------------------------------------------------
    # association matrix
    # -------------------------------------------------
    def get_association_matrix(self, query, key=None, mask=None):

        if key is None:
            key = query

        if self.use_layer_norm:
            query = self.norm_query(query)
            key = self.norm_key(key)

        if self.use_projection:
            q = self.q_proj(query)
            k = self.k_proj(key)
        else:
            q = query
            k = key

        q = self.split_heads(q)
        k = self.split_heads(k)

        scores = torch.matmul(q, k.transpose(-2, -1))

        beta = self._get_beta()

        scores = scores * beta / math.sqrt(self.head_dim)

        scores = self._apply_mask(scores, mask)

        attn = F.softmax(scores, dim=-1)

        return attn

    # -------------------------------------------------
    # Hopfield energy function
    # -------------------------------------------------
    def compute_energy(self, query, key=None):

        if key is None:
            key = query

        if self.use_layer_norm:
            query = self.norm_query(query)
            key = self.norm_key(key)

        if self.use_projection:
            q = self.q_proj(query)
            k = self.k_proj(key)
        else:
            q = query
            k = key

        q = self.split_heads(q)
        k = self.split_heads(k)

        scores = torch.matmul(q, k.transpose(-2, -1))

        beta = self._get_beta()

        scores = scores * beta / math.sqrt(self.head_dim)

        lse = torch.logsumexp(scores, dim=-1)

        energy = -lse / beta.squeeze(-1)

        return energy

    # -------------------------------------------------
    def reset_parameters(self):

        if self.use_projection:
            nn.init.xavier_uniform_(self.q_proj.weight)
            nn.init.xavier_uniform_(self.k_proj.weight)
            nn.init.xavier_uniform_(self.v_proj.weight)
            nn.init.xavier_uniform_(self.out_proj.weight)

            nn.init.zeros_(self.q_proj.bias)
            nn.init.zeros_(self.k_proj.bias)
            nn.init.zeros_(self.v_proj.bias)
            nn.init.zeros_(self.out_proj.bias)

        if self.use_layer_norm:
            self.norm_query.reset_parameters()
            self.norm_key.reset_parameters()
            self.norm_value.reset_parameters()

        nn.init.constant_(self.beta_param, 1.0)